# 📊 Laboratorio de Streamlit: el dashboard de mi portafolio
**Lenguajes de Programación · Ingeniería Financiera · ITM · Google Colab**

> **Reto:** convertir el análisis del taller *"Del análisis exploratorio a la presentación de un portafolio"* en una aplicación web que le muestre a un inversionista **las bondades de tu portafolio**.

**Al finalizar podrás:** organizar tu análisis en una página web con barra lateral, columnas y pestañas; mostrar indicadores con `st.metric`; reutilizar tus gráficos de `matplotlib` y `seaborn`; agregar controles para el usuario y publicar la aplicación.

**No hay teoría nueva.** Todo lo que verás en el dashboard ya lo calculaste en la guía de EDA y en el taller: rendimientos, valor de la inversión, volatilidad, Sharpe, correlación, VaR y *drawdown*. Aquí solo aprendes a **presentarlo**.

---

### 🧭 Cómo trabajar este notebook

1. Guarda una copia en tu Drive: **Archivo → Guardar una copia en Drive**.
2. Ejecuta las celdas **en orden**.
3. Las celdas que empiezan con `%%writefile` **crean o reemplazan un archivo completo**. Después de cada versión de la app se guarda una copia de respaldo.
4. El servidor de Streamlit se inicia **una sola vez** y detecta solo los cambios en los archivos.

| Ícono | Significado |
|---|---|
| 🧩 | **Modifica**: cambia algo en tus archivos |
| ✍️ | **Responde** o interpreta |
| ✅ | **Verificación**: si no hay error, vas bien |
| 💼 | **Pensando en el inversionista** |

> **Importante:** Colab ejecuta Python en un equipo remoto. La app se ve en **otra pestaña** del navegador. No necesitas instalar nada en tu computador.

### 🗂️ La idea central: dos archivos

Para que sea simple, la aplicación se divide en **dos archivos**:

| Archivo | Qué contiene | ¿Quién lo edita? |
|---|---|---|
| `mi_portafolio.py` | **Tu contenido**: activos, pesos, benchmark, fechas y los textos para el inversionista (secciones 1 y 13 del taller) | **Tú** |
| `app.py` | **La plantilla del dashboard**: descarga datos, calcula y dibuja | Lo construimos juntos paso a paso |

Así, para mostrar *tu* portafolio basta con llenar `mi_portafolio.py`. Ningún equipo tiene que reescribir el dashboard desde cero.

### 🔁 Del taller al dashboard

El dashboard responde las mismas preguntas de la sección 12 del taller, **una pestaña por pregunta**:

| Pregunta del inversionista | Pestaña | Viene del taller | Elemento de Streamlit |
|---|---|---|---|
| ¿Qué tengo? | 🧺 | Sección 1 | `st.dataframe`, `st.pyplot` |
| ¿Cuánto he ganado? | 💰 | Secciones 3 y 4 | `st.line_chart`, `st.multiselect` |
| ¿Cuánto riesgo asumo? | ⚖️ | Sección 6 | `st.pyplot` (mapa riesgo–rendimiento) |
| ¿Estoy diversificado? | 🔗 | Sección 7 | `st.pyplot` (heatmap), `st.metric` |
| ¿Cuánto puedo perder? | 📉 | Secciones 8, 10 y 11 | `st.pyplot`, `st.area_chart` |
| ¿Qué concluyen ustedes? | 📝 | Secciones 11 y 13 | `st.write`, `st.dataframe` |

## 01 · Preparar el entorno

**¿Qué es Streamlit?** Una librería de Python que convierte instrucciones como `st.title()` o `st.metric()` en componentes de una página web.

| En el notebook del taller | En Streamlit |
|---|---|
| `print(f"Volatilidad: {vol:.2%}")` | `st.metric("Volatilidad", f"{vol:.2%}")` |
| `plt.show()` | `st.pyplot(fig)` |
| `display(tabla)` | `st.dataframe(tabla)` |
| Cambias `monto_inicial` editando el código | El usuario lo cambia con un control |

In [1]:
%pip -q install "streamlit>=1.40,<2" "yfinance>=0.2.50"

In [2]:
import sys, streamlit, pandas, numpy, matplotlib, seaborn, scipy, yfinance
print("Python    :", sys.version.split()[0])
print("Streamlit :", streamlit.__version__)
print("Pandas    :", pandas.__version__)
print("yfinance  :", yfinance.__version__)

Python    : 3.13.18
Streamlit : 1.67.0
Pandas    : 2.2.3
yfinance  : 0.2.66


## 02 · Tu ficha: `mi_portafolio.py`

### 🧩 2.1 Crea el archivo con los datos de tu portafolio

Reemplaza los valores de ejemplo por **los de tu taller**:

- **Sección 1 del taller →** `nombre_portafolio`, `activos`, `info_activos`, `pesos`, `benchmark`, fechas, `monto_inicial`.
- **Sección 13 del taller →** `justificacion`, `resumen_ejecutivo`, `hallazgos`, `riesgos`, `limitaciones`, `perfil_inversionista`.

> 💡 Puedes completar los textos al final, cuando ya veas tus resultados en el dashboard. Ejecuta esta celda **una sola vez**; después edita el archivo desde el panel **Archivos** de Colab (ver sección 04).

In [3]:
%%writefile mi_portafolio.py
# =====================================================================
#  mi_portafolio.py  ·  LA FICHA DE TU PORTAFOLIO
#  Copia aquí los valores de la sección 1 del taller y tus respuestas
#  de la sección 13. La app (app.py) lee este archivo: no necesitas
#  tocar app.py para que el dashboard muestre TU portafolio.
# =====================================================================

# ---------- 1. Identificación ----------------------------------------
nombre_portafolio = "2T_VALORA."
integrantes = "Diego German Andrade Jimenez"

# ---------- 2. Composición (igual que en el taller) -------------------
activos = ["CEG", "LFUS", "TECK", "SNDR", "STRL", "POWL", "EME", "VRT"]

# Nombre y sector de cada activo (se usará en tablas y en el dashboard)
info_activos = {
    # "AAPL": ("Apple", "Tecnología"),
    "CEG": ("Constellation Energy Corporation", "Utilities"),
    "LFUS": ("Littelfuse.Inc.", "Tecnología"),
    "TECK": ("Teck Resources Limited", "Materiales"),
    "SNDR": ("Schneider National.Inc.", "Industriales"),
    "STRL": ("Sterling Infrastructure.Inc.", "Industriales"),
    "POWL": ("Powell Industries.Inc.", "Industriales"),
    "EME": ("EMCOR Group.Inc.", "Industriales"),
    "VRT": ("Vertiv Holdings.Co.", "Industriales"),
}

# Pesos en el MISMO orden de `activos`. Deben sumar 1.
pesos = [0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125, 0.125]

benchmark = "VOO"
fecha_inicio = "2023-01-01"
fecha_fin = "2026-11-20"
monto_inicial = 10_00
tasa_libre_riesgo = 0.045      # anual

# ---------- 3. Mensaje para el inversionista (sección 13 del taller) --
justificacion = (
    "Eleccion de integrantes enfocada en la Industria 5.0 impulsada por el desarrollo de la IA. Empresas determinan el CAPEX y BACKLOG de la industria IA. Los pesos para este rebalanceo que minimizan el CVaR y Maximiza el Rendimiento"
)

resumen_ejecutivo = (
    " CEG para el suministro electrico (Nuclear y Renovables). VRT para el Equipamiento eléctrico, gestión térmica y centros de datos. "
    " EME aporta Ingeniería y construcción mecánica y eléctrica. POWL Equipos y aparamenta eléctrica de distribución. "
    " STRL promueve la Ingeniería, obra civil y preparación de terrenos. NVT Protección eléctrica y refrigeración líquida. "
    " FIX construida para desarrollar Ingeniería, climatización (HVAC) e instalación eléctrica. LFUS Componentes electrónicos y protección de circuitos. "
    " TECK Básicos	Minería de metales base (Cobre y zinc). SNDR Transporte de carga, intermodal y logística."
    " Portafolio que impulsa el desarrollo tecnologico, evidentemente con mayor volatilidad que el benchmark pero incluso mayores rendimientos superandolo en Sharpe CVaR y VaR."
)

hallazgos = [
    "Rendimiento: 37.87% EA",
    "Riesgo y diversificación: Volatilidad anualizada 32-14%. prima por diversificar = -14-6% del riesgo ",
    "Eventos extremos: Maxima Caida = -38.51%. VaR Diario(Historico) = -3.16%. Peor Dia = 12.5%",
    "la valoracion de cada empresa que integra el portafolio tiene mejores multiplos que la media del sector",
]

riesgos = "Principales riesgos: concentración = altamente industrial, correlaciones = positiva en 50 puntos, colas pesadas = "

limitaciones = (
    "Limitaciones: período analizado, rendimientos pasados frente a futuros, "
    "supuesto de normalidad, rebalanceo diario, costos de transacción..."
)

perfil_inversionista = "Agresivo"   # Conservador, Moderado o Agresivo




Writing mi_portafolio.py


### ✅ 2.2 Verificación

Las mismas reglas del taller. Si aparece un error, corrige `mi_portafolio.py` y vuelve a ejecutar **esta** celda.

In [4]:
import importlib, numpy as np
import mi_portafolio as mp
importlib.reload(mp)   # vuelve a leer el archivo si lo modificaste

assert 5 <= len(mp.activos) <= 8, "El portafolio debe tener entre 5 y 8 activos."
assert len(mp.pesos) == len(mp.activos), "Debe haber un peso por cada activo."
assert np.isclose(sum(mp.pesos), 1), f"Los pesos deben sumar 1. Ahora suman {sum(mp.pesos):.4f}"
assert min(mp.pesos) >= 0, "No se permiten pesos negativos."
assert set(mp.info_activos) == set(mp.activos), "info_activos debe tener exactamente los mismos tickers."
assert len({s for _, s in mp.info_activos.values()}) >= 3, "Incluye al menos 3 sectores."
assert mp.benchmark not in mp.activos, "El benchmark no debe ser uno de los activos."
assert len(mp.hallazgos) == 3, "Escribe exactamente tres hallazgos."
print(f"✔ '{mp.nombre_portafolio}' bien definido: {len(mp.activos)} activos + benchmark {mp.benchmark}")

AssertionError: Escribe exactamente tres hallazgos.

### 2.3 Guardar una copia de los precios (respaldo)

La app descarga los precios de Yahoo Finance cada vez que se abre. Pero Yahoo **a veces no responde** (sobre todo desde servidores en la nube, como Streamlit Community Cloud). Por eso guardamos ahora una copia en `precios_respaldo.csv`: si la descarga falla, la app usa este archivo **y lo avisa en pantalla**.

> 🔁 Si cambias tus activos o fechas en `mi_portafolio.py`, vuelve a ejecutar las celdas 2.2 y 2.3.

In [5]:
import yfinance as yf

tickers = mp.activos + [mp.benchmark]
datos = yf.download(tickers, start=mp.fecha_inicio, end=mp.fecha_fin,
                    auto_adjust=True, progress=False)
cierre_total = datos["Close"][tickers].dropna()
cierre_total.to_csv("precios_respaldo.csv")

print(f"✔ precios_respaldo.csv guardado: {cierre_total.shape[0]} días, "
      f"del {cierre_total.index[0].date()} al {cierre_total.index[-1].date()}")
cierre_total.tail(3)

✔ precios_respaldo.csv guardado: 975 días, del 2023-01-03 al 2026-11-19


Ticker,CEG,LFUS,TECK,SNDR,STRL,POWL,EME,VRT,VOO
Date,,,,,,,,,
2026-11-17,268.070007,378.645050,56.080002,28.780001,275.260010,150.770004,646.239990,190.369995,673.190002
2026-11-18,262.739990,385.909973,52.430000,28.940001,276.359985,154.949997,646.229980,194.479996,671.309998
2026-11-19,252.029999,383.869995,51.580002,28.520000,273.250000,158.750000,636.080017,184.990005,658.010010


## 03 · Primera versión: el esqueleto del dashboard

**Layout** es la distribución de los componentes en la pantalla. Estos son los únicos elementos que necesitas para empezar:

| Instrucción | Qué hace |
|---|---|
| `import mi_portafolio as mp` | Trae tu ficha. Luego usas `mp.activos`, `mp.pesos`, etc. |
| `st.set_page_config(layout="wide")` | Usa todo el ancho de la pantalla |
| `st.title()`, `st.caption()` | Título y subtítulo |
| `st.sidebar.…` | Todo lo que empiece así va en el **panel lateral** |
| `c1, c2, c3 = st.columns(3)` | Divide la página en tres columnas |
| `st.metric("Nombre", valor)` | Muestra un indicador grande |
| `t1, t2 = st.tabs([...])` + `with t1:` | Crea pestañas y dice qué va en cada una |
| `st.dataframe(tabla)` | Muestra una tabla de pandas |

**Antes de ejecutar:** predice dónde aparecerá cada elemento. Luego compruébalo en el navegador.

In [6]:
%%writefile app.py
import pandas as pd
import streamlit as st

import mi_portafolio as mp   # <- tu ficha: activos, pesos, textos

st.set_page_config(page_title=mp.nombre_portafolio, page_icon="📊", layout="wide")
st.write()


Writing app.py


## 04 · Iniciar Streamlit y abrir la aplicación

### 4.1 Iniciar el servidor

El servidor se ejecuta **en segundo plano** para que la celda termine. Luego verificamos que responda en `http://127.0.0.1:8501/_stcore/health`.

**No avances al paso 4.2 si no aparece `✅ ... 200 ok`.**

In [7]:
import subprocess, time, requests
from pathlib import Path


# Evita dejar varios servidores compitiendo por el puerto 8501
subprocess.run(["pkill", "-f", "[s]treamlit run app.py"], check=False)
log_streamlit = open("/tmp/streamlit.log", "w")
servidor = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0",
     "--server.headless=true", "--server.fileWatcherType=poll"],
    stdout=log_streamlit, stderr=subprocess.STDOUT, start_new_session=True
)

for intento in range(20):
    try:
        respuesta = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=2)
        if respuesta.status_code == 200:
            print("✅ Streamlit funcionando:", respuesta.status_code, respuesta.text)
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    print("❌ Streamlit no responde. Últimos mensajes:")
    print(Path("/tmp/streamlit.log").read_text(errors="replace")[-3000:])

✅ Streamlit funcionando: 200 ok


### 4.2 Abrir la aplicación en el navegador

Usamos un **túnel temporal de Cloudflare** que crea una dirección pública hacia tu sesión de Colab. El enlace funciona mientras la sesión de Colab esté abierta.

In [8]:
import platform, urllib.request, os
from pathlib import Path

arquitectura = platform.machine().lower()
archivo = {"x86_64": "cloudflared-linux-amd64", "aarch64": "cloudflared-linux-arm64"}.get(arquitectura)
if not archivo:
    raise RuntimeError(f"Arquitectura no contemplada: {arquitectura}")
if not Path("/tmp/cloudflared").exists():
    urllib.request.urlretrieve(
        f"https://github.com/cloudflare/cloudflared/releases/latest/download/{archivo}", "/tmp/cloudflared")
os.chmod("/tmp/cloudflared", 0o755)
print("✔ cloudflared preparado")

✔ cloudflared preparado


In [9]:
import re, subprocess, time
from pathlib import Path

subprocess.run(["pkill", "-f", "[c]loudflared tunnel --url"], check=False)
registro = open("/tmp/cloudflare.log", "w")
tunel = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
    stdout=registro, stderr=subprocess.STDOUT, start_new_session=True
)

url_publica = None
for intento in range(25):
    time.sleep(1)
    texto = Path("/tmp/cloudflare.log").read_text(errors="replace")
    encontrado = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", texto)
    if encontrado:
        url_publica = encontrado.group(0)
        break

if url_publica:
    print("🌐 ABRE TU DASHBOARD EN UNA PESTAÑA NUEVA:", url_publica)
else:
    print("No se obtuvo la dirección. Diagnóstico:")
    print(texto[-2500:])

🌐 ABRE TU DASHBOARD EN UNA PESTAÑA NUEVA: https://bool-lance-selecting-aggregate.trycloudflare.com


**Comprobación visual:** debe verse el nombre de tu portafolio, la composición en la barra lateral, tres indicadores y tres pestañas.

Si la página no carga o aparece `502 Bad Gateway`, espera unos segundos y recarga. Si continúa, ejecuta el diagnóstico:

In [ ]:
from pathlib import Path
import requests
try:
    r = requests.get("http://127.0.0.1:8501/_stcore/health", timeout=5)
    print("Servidor local:", r.status_code, r.text)
except Exception as e:
    print("Servidor local sin respuesta:", e)
for nombre, ruta in [("Streamlit", "/tmp/streamlit.log"), ("Cloudflare", "/tmp/cloudflare.log")]:
    texto = Path(ruta).read_text(errors="replace")[-1200:] if Path(ruta).exists() else "(aún no existe)"
    print(f"\nRegistro {nombre}:\n", texto)

Servidor local: 200 ok

Registro Streamlit:
 

2026-11-20 17:47:06.480 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.21.155.15:8501



Registro Cloudflare:
 S Resolution" details="DNS Resolved successfully" run_id=25856590-4d94-40ad-82f1-c2aaa437f599 status=pass target=region2.v2.argotunnel.com
2026-11-20T17:47:46Z INF precheck component="UDP Connectivity" details="QUIC connection successful" run_id=25856590-4d94-40ad-82f1-c2aaa437f599 status=pass target=region1.v2.argotunnel.com
2026-11-20T17:47:46Z INF precheck component="UDP Connectivity" details="QUIC connection successful" run_id=25856590-4d94-40ad-82f1-c2aaa437f599 status=pass target=region2.v2.argotunnel.com
2026-11-20T17:47:46Z INF precheck component="TCP Connectivity" details="HTTP/2 connection successful" run_id=25856590-4d94-40ad-82f1-c2aaa437f599 status=pass target=region1.v2.

### 🧩 4.3 Edita un archivo y mira el cambio en vivo

1. En Colab abre el ícono de **Archivos** 📁 (panel izquierdo) y haz doble clic en `mi_portafolio.py`.
2. Cambia `nombre_portafolio` e `integrantes`. Guarda con `Ctrl+S` / `Cmd+S`.
3. Vuelve a la pestaña del dashboard: arriba a la derecha aparecerá **Rerun** (o pulsa la tecla `R`).

> ⚠️ **No vuelvas a ejecutar la celda `%%writefile mi_portafolio.py`**: reemplazaría tus cambios por el ejemplo.

**Concepto clave:** cada vez que algo cambia (un archivo o un control), Streamlit **vuelve a ejecutar `app.py` de arriba abajo**. Por eso el código de la app se lee como un notebook en una sola celda.

## 05 · Segunda versión: datos reales e indicadores clave

Ahora la app descarga los precios y calcula, **con el mismo código del taller**, el valor de la inversión, el rendimiento y la volatilidad anualizados.

Tres ideas nuevas:

**1. `@st.cache_data`** — Como Streamlit re-ejecuta todo el script con cada clic, sin caché descargaría los precios una y otra vez. El decorador guarda el resultado: la primera vez descarga y las siguientes reutiliza (durante una hora, `ttl=3600`).

**2. `try / except`** — Si Yahoo Finance falla, la app usa `precios_respaldo.csv` y lo indica en la barra lateral. **Nunca presentes datos sin decir de dónde vienen.**

**3. `st.metric(nombre, valor, delta)`** — El tercer argumento muestra una comparación con flecha y color. Lo usamos para comparar contra el benchmark:
- un rendimiento mayor que el del benchmark sale en **verde**;
- en la volatilidad usamos `delta_color="inverse"`: **más volatilidad sale en rojo**, porque es más riesgo.

### ✅ Comprueba con el taller

Los tres indicadores deben coincidir con los de tu notebook del taller (secciones 4.3 y 6.1) si usas las mismas fechas. Si no coinciden, revisa `activos`, `pesos` y fechas en `mi_portafolio.py`.

### ✍️ Preguntas

1. ¿Qué significa que el *delta* del rendimiento salga en verde? ¿Y que el de la volatilidad salga en rojo?
2. ¿Qué fuente de datos aparece en la barra lateral? ¿Por qué es importante mostrarla?

**Respuestas:**

## 06 · Versión final: el dashboard completo

Esta versión agrega las pestañas de riesgo, diversificación y pérdidas, además de **controles** en la barra lateral.

### 6.1 Tus gráficos del taller funcionan casi igual

Para llevar una figura de `matplotlib` o `seaborn` a Streamlit **solo cambia la última línea**:

```python
# En el taller                         # En el dashboard
fig, ax = plt.subplots(...)            fig, ax = plt.subplots(...)
sns.heatmap(..., ax=ax)                sns.heatmap(..., ax=ax)
plt.show()                             st.pyplot(fig)
                                       plt.close(fig)   # libera memoria
```

Para las series de tiempo usamos `st.line_chart` y `st.area_chart`, que son **interactivos** (el inversionista pasa el mouse y ve el valor de cada día).

### 6.2 Controles en la barra lateral

| Control | Qué cambia en el dashboard |
|---|---|
| `st.sidebar.slider` (dos fechas) | El período analizado: **todos** los indicadores y gráficos se recalculan |
| `st.sidebar.number_input` | El monto inicial de la inversión |
| `st.sidebar.selectbox` | El nivel de confianza del VaR: 95% o 99% |
| `st.multiselect` (pestaña 💰) | Qué activos se ven en el gráfico base 100 |

El valor que devuelve cada control es **una variable normal de Python**. Por ejemplo, `confianza` vale `0.95` o `0.99`, y el VaR se calcula con `rend_port.quantile(1 - confianza)`, exactamente como en el taller.

### 6.3 Cómo está organizado `app.py`

Lee el código buscando estos encabezados: **0.** configuración · **1.** datos · **2.** barra lateral · **3.** cálculos (los del taller) · **4.** indicadores · **5.** pestañas · **6.** datos y nota legal.

In [10]:
%%writefile app.py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import streamlit as st
import yfinance as yf
from pathlib import Path
from scipy import stats
from scipy.stats import norm

import mi_portafolio as mp   # <- tu ficha: activos, pesos, textos
from scipy.optimize import minimize

# =====================================================================
# 0. CONFIGURACIÓN DE LA PÁGINA
# =====================================================================
st.set_page_config(page_title=mp.nombre_portafolio, page_icon="📊", layout="wide")
sns.set_theme(style="whitegrid")
DIAS_ANIO = 252

activos = mp.activos
benchmark = mp.benchmark
pesos = pd.Series(mp.pesos, index=activos, name="Peso")

st.title(f"📊 {mp.nombre_portafolio}")
st.caption(f"Dashboard para inversionistas · {mp.integrantes}")

# =====================================================================
# 1. DATOS: Yahoo Finance (con archivo de respaldo)
# =====================================================================
@st.cache_data(ttl=3600, show_spinner="Descargando precios de Yahoo Finance...")
def descargar_precios(tickers, inicio, fin):
    datos = yf.download(list(tickers), start=inicio, end=fin,
                        auto_adjust=True, progress=False)
    cierre = datos["Close"][list(tickers)].dropna()
    if len(cierre) < 30:
        raise ValueError("Yahoo Finance devolvió muy pocos datos")
    return cierre

tickers = activos + [benchmark]
try:
    cierre_total = descargar_precios(tuple(tickers), mp.fecha_inicio, mp.fecha_fin)
    fuente = "Yahoo Finance"
except Exception:
    if Path("precios_respaldo.csv").exists():
        cierre_total = pd.read_csv("precios_respaldo.csv", index_col=0, parse_dates=True)
        cierre_total = cierre_total[tickers].dropna()
        fuente = "archivo precios_respaldo.csv (Yahoo Finance no respondió)"
    else:
        st.error("No fue posible descargar los precios y no existe precios_respaldo.csv.")
        st.stop()

# =====================================================================
# 2. BARRA LATERAL: controles del usuario
# =====================================================================


st.sidebar.header("⚙️ Ajusta el análisis")

primera = cierre_total.index[0].date()
ultima = cierre_total.index[-1].date()
inicio, fin = st.sidebar.slider("Período analizado", min_value=primera, max_value=ultima,
                                value=(primera, ultima), format="YYYY-MM-DD")
monto_inicial = st.sidebar.number_input("Monto inicial (USD)", min_value=1_000,
                                        value=int(mp.monto_inicial), step=1_000)
confianza = st.sidebar.selectbox("Nivel de confianza del VaR", [0.95, 0.99],
                                 format_func=lambda x: f"{x:.0%}")

st.sidebar.divider()
st.sidebar.subheader("🧺 Composición")
#st.sidebar.dataframe(pesos.map("{:.0%}".format))
#st.sidebar.caption(f"Benchmark: **{benchmark}** · Fuente: {fuente}")

cierre_periodo = cierre_total.loc[str(inicio):str(fin)]
if len(cierre_periodo) < 60:
    st.warning("Elige un período más largo (al menos 60 días de negociación).")
    st.stop()

cierre = cierre_periodo[activos]
cierre_bench = cierre_periodo[benchmark]

# =====================================================================
# 3. CÁLCULOS (los mismos del taller)
# =====================================================================
# Rendimientos de los activos
rend_simple = cierre.pct_change().dropna()
rendimientos = np.log(cierre / cierre.shift(1)).dropna()

# optimizacion del CVaR, Portafolio con pesos eficientes
def optimizar_pesos_cvar(retornos_sub, pesos_minimos=None, max_peso=0.15):
    """Minimiza el CVaR histórico al 95% de los activos seleccionados."""
    n = retornos_sub.shape[1]
    if n == 0:
        return pd.Series(dtype=float)
    if n == 1:
        return pd.Series([min(1.0, max_peso)], index=retornos_sub.columns)

    R = retornos_sub.fillna(0).values
    alpha = 0.05

    def objetivo_cvar(w):
        port_r = R @ w
        var_95 = np.percentile(port_r, alpha * 100)
        cola = port_r[port_r <= var_95]
        cvar = -np.mean(cola) if len(cola) > 0 else -var_95
        # Penalización suave por concentración para fomentar diversificación
        return cvar + 0.05 * np.sum(w ** 2)

    w0 = np.ones(n) / n
    bounds = [(0.02, max_peso) for _ in range(n)]
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    res = minimize(objetivo_cvar, w0, method="SLSQP", bounds=bounds, constraints=constraints)
    if res.success:
        w_opt = res.x / np.sum(res.x)
    else:
        w_opt = w0
    return pd.Series(w_opt, index=retornos_sub.columns)

w_opt = optimizar_pesos_cvar(rendimientos, max_peso=0.15)
print(w_opt)
pesos=w_opt
st.sidebar.dataframe(pesos.map("{:.0%}".format))
st.sidebar.caption(f"Benchmark: **{benchmark}** · Fuente: {fuente}")

#st.draw()



# Portafolio (rebalanceo diario) y benchmark
rend_port = np.log(1 + (rend_simple * pesos).sum(axis=1))
rend_port.name = "Portafolio"
rend_bench = np.log(cierre_bench / cierre_bench.shift(1)).dropna()
rend_bench.name = benchmark
rend_todos = pd.concat([rendimientos, rend_port, rend_bench], axis=1)

# Valor de la inversión (arranca en el monto inicial)
fecha0 = cierre.index[0]
valor_port = pd.concat([pd.Series([monto_inicial], index=[fecha0]),
                        monto_inicial * np.exp(rend_port.cumsum())])
valor_bench = pd.concat([pd.Series([monto_inicial], index=[fecha0]),
                         monto_inicial * np.exp(rend_bench.cumsum())])

# Resumen anualizado de cada serie
resumen = pd.DataFrame({
    "Rend. anualizado": rend_todos.mean() * DIAS_ANIO,
    "Volatilidad anualizada": rend_todos.std() * np.sqrt(DIAS_ANIO),
})

def max_drawdown(valor):
    return (valor / valor.cummax() - 1).min()

def var_dinero(cuantil_log, valor):
    return valor * (1 - np.exp(cuantil_log))

def ficha(r, valor):
    mu_a = r.mean() * DIAS_ANIO
    sigma_a = r.std() * np.sqrt(DIAS_ANIO)
    return {
        "Rendimiento total": valor.iloc[-1] / valor.iloc[0] - 1,
        "Rendimiento anualizado": mu_a,
        "Volatilidad anualizada": sigma_a,
        "Sharpe": (mu_a - mp.tasa_libre_riesgo) / sigma_a,
        "Máxima caída": max_drawdown(valor),
        "VaR diario (histórico)": r.quantile(1 - confianza),
        "Peor día": r.min(),
        "Mejor día": r.max(),
        "Asimetría": stats.skew(r),
        "Exceso de curtosis": stats.kurtosis(r),
    }

ficha_tecnica = pd.DataFrame({"Portafolio": ficha(rend_port, valor_port),
                              benchmark: ficha(rend_bench, valor_bench)})
fp, fb = ficha_tecnica["Portafolio"], ficha_tecnica[benchmark]

# Parámetros de la normal y VaR en dinero
mu_p, sigma_p = rend_port.mean(), rend_port.std()
q_hist = rend_port.quantile(1 - confianza)
q_norm = norm.ppf(1 - confianza, loc=mu_p, scale=sigma_p)
valor_actual = valor_port.iloc[-1]

# =====================================================================
# 4. INDICADORES CLAVE (st.metric)
# =====================================================================
st.subheader("Indicadores clave del período")
c1, c2, c3, c4, c5 = st.columns(5)
c1.metric("Valor final", f"USD {valor_actual:,.0f}",
          f"{valor_actual - valor_bench.iloc[-1]:+,.0f} vs {benchmark}")
c2.metric("Rendimiento anualizado", f"{fp['Rendimiento anualizado']:.1%}",
          f"{(fp['Rendimiento anualizado'] - fb['Rendimiento anualizado']) * 100:+.1f} pp vs {benchmark}")
c3.metric("Volatilidad anualizada", f"{fp['Volatilidad anualizada']:.1%}",
          f"{(fp['Volatilidad anualizada'] - fb['Volatilidad anualizada']) * 100:+.1f} pp vs {benchmark}",
          delta_color="inverse")
c4.metric("Sharpe", f"{fp['Sharpe']:.2f}", f"{fp['Sharpe'] - fb['Sharpe']:+.2f} vs {benchmark}")
c5.metric(f"VaR diario {confianza:.0%}", f"USD {var_dinero(q_hist, valor_actual):,.0f}",
          f"{q_hist:.2%} del valor", delta_color="off")
st.caption("Más volatilidad aparece en rojo porque significa más riesgo. "
           "Portafolio con pesos fijos y rebalanceo diario; sin costos de transacción.")

# =====================================================================
# 5. PESTAÑAS: una pregunta del inversionista por pestaña
# =====================================================================
t1, t2, t3, t4, t5, t6 = st.tabs(["🧺 ¿Qué tengo?", "💰 ¿Cuánto he ganado?",
                                  "⚖️ ¿Cuánto riesgo?", "🔗 ¿Estoy diversificado?",
                                  "📉 ¿Cuánto puedo perder?", "📝 Nuestro mensaje"])

# ---------- Pestaña 1: composición -----------------------------------
with t1:
    izq, der = st.columns(2)
    with izq:
        tabla = pd.DataFrame(mp.info_activos, index=["Nombre", "Sector"]).T
        tabla["Peso"] = pesos.map("{:.0%}".format)
        st.dataframe(tabla)
        st.markdown(f"**¿Por qué este portafolio?** {mp.justificacion}")
    with der:
        por_sector = tabla.assign(Peso=pesos).groupby("Sector")["Peso"].sum().sort_values()
        fig, ax = plt.subplots(figsize=(6, 3.5))
        ax.barh(por_sector.index, por_sector.values, color="steelblue", edgecolor="black")
        for i, v in enumerate(por_sector.values):
            ax.text(v + 0.005, i, f"{v:.0%}", va="center")
        ax.set_title("Peso por sector", fontweight="bold")
        ax.set_xlabel("Peso")
        st.pyplot(fig)
        plt.close(fig)

# ---------- Pestaña 2: rendimiento -----------------------------------
with t2:
    st.markdown(f"**Valor de una inversión de USD {monto_inicial:,.0f}**")
    st.line_chart(pd.DataFrame({mp.nombre_portafolio: valor_port, benchmark: valor_bench}))
    st.info(f"Una inversión de USD {monto_inicial:,.0f} el {inicio} se habría convertido en "
            f"USD {valor_actual:,.0f} el {cierre.index[-1].date()}, frente a "
            f"USD {valor_bench.iloc[-1]:,.0f} en {benchmark}.")

    st.markdown("**Evolución de cada activo (inicio = 100)**")
    elegidos = st.multiselect("Activos a mostrar", activos, default=activos)
    base_100 = cierre[elegidos] / cierre[elegidos].iloc[0] * 100
    base_100[f"{benchmark} (benchmark)"] = cierre_bench / cierre_bench.iloc[0] * 100
    st.line_chart(base_100)

# ---------- Pestaña 3: riesgo-rendimiento ----------------------------
with t3:
    izq, der = st.columns([3, 2])
    with izq:
        fig, ax = plt.subplots(figsize=(8, 5.5))
        for nombre in resumen.index:
            x = resumen.loc[nombre, "Volatilidad anualizada"]
            y = resumen.loc[nombre, "Rend. anualizado"]
            if nombre == "Portafolio":
                ax.scatter(x, y, s=300, color="navy", marker="*", zorder=3)
            elif nombre == benchmark:
                ax.scatter(x, y, s=200, color="black", marker="D", zorder=3)
            else:
                ax.scatter(x, y, s=120, color="steelblue", edgecolor="black", zorder=3)
            ax.annotate(nombre, (x, y), xytext=(8, 5), textcoords="offset points", fontweight="bold")
        ax.axhline(0, color="black", linewidth=1)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
        ax.set_title("Mapa riesgo–rendimiento (anualizado)", fontweight="bold")
        ax.set_xlabel("Volatilidad anualizada (riesgo)")
        ax.set_ylabel("Rendimiento anualizado")
        st.pyplot(fig)
        plt.close(fig)
    with der:
        st.dataframe(resumen.style.format("{:.2%}"))
        st.caption("Arriba y a la izquierda = más rendimiento con menos riesgo.")

# ---------- Pestaña 4: diversificación -------------------------------
with t4:
    correlacion = rendimientos.corr()
    corr_promedio = correlacion.values[np.triu_indices_from(correlacion, k=1)].mean()
    vol_activos = rendimientos.std() * np.sqrt(DIAS_ANIO)
    vol_sin_div = (pesos * vol_activos).sum()
    vol_port = fp["Volatilidad anualizada"]

    izq, der = st.columns([3, 2])
    with izq:
        fig, ax = plt.subplots(figsize=(7, 5.5))
        sns.heatmap(correlacion, annot=True, fmt=".2f", cmap="coolwarm",
                    center=0, vmin=-1, vmax=1, square=True, ax=ax)
        ax.set_title("Correlación entre los rendimientos", fontweight="bold")
        st.pyplot(fig)
        plt.close(fig)
    with der:
        st.metric("Correlación promedio", f"{corr_promedio:.2f}")
        st.metric("Volatilidad si ρ = 1", f"{vol_sin_div:.1%}")
        st.metric("Volatilidad real del portafolio", f"{vol_port:.1%}",
                  f"{(vol_port - vol_sin_div) * 100:+.1f} pp por diversificar", delta_color="inverse")
        st.success(f"La diversificación reduce el riesgo en un "
                   f"{(vol_sin_div - vol_port) / vol_sin_div:.0%} frente a un portafolio "
                   f"con activos perfectamente correlacionados.")

# ---------- Pestaña 5: pérdidas --------------------------------------
with t5:
    izq, der = st.columns(2)
    with izq:
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.hist(rend_port, bins=60, density=True, color="steelblue", alpha=0.5, edgecolor="black")
        xs = np.linspace(rend_port.min(), rend_port.max(), 400)
        ax.plot(xs, norm.pdf(xs, mu_p, sigma_p), color="red", linewidth=2, label="Normal ajustada")
        ax.axvline(q_hist, color="darkred", linestyle="--", linewidth=2,
                   label=f"VaR {confianza:.0%} histórico: {q_hist:.2%}")
        ax.set_title("Rendimientos diarios del portafolio", fontweight="bold")
        ax.set_xlabel("Rendimiento diario")
        ax.legend(fontsize=9)
        st.pyplot(fig)
        plt.close(fig)
    with der:
        var_tabla = pd.DataFrame({
            "Rendimiento (%)": [q_norm, q_hist],
            "Pérdida (USD)": [var_dinero(q_norm, valor_actual), var_dinero(q_hist, valor_actual)],
        }, index=["VaR normal", "VaR histórico"])
        st.dataframe(var_tabla.style.format({"Rendimiento (%)": "{:.2%}", "Pérdida (USD)": "{:,.2f}"}))
        st.info(f"Con un {confianza:.0%} de confianza, la pérdida del portafolio en un día "
                f"no debería superar USD {var_dinero(q_hist, valor_actual):,.0f} "
                f"({-q_hist:.2%} de su valor), según la historia del período.")

    st.markdown("**Caída desde el máximo previo (drawdown)**")
    dd = pd.DataFrame({"Portafolio": valor_port / valor_port.cummax() - 1,
                       benchmark: valor_bench / valor_bench.cummax() - 1}) * 100
    st.area_chart(dd, y_label="Drawdown (%)", stack=False)
    st.caption(f"Máxima caída del portafolio: {fp['Máxima caída']:.1%} "
               f"(benchmark: {fb['Máxima caída']:.1%}).")

# ---------- Pestaña 6: mensaje ---------------------------------------
with t6:
    st.subheader("Resumen ejecutivo")
    st.write(mp.resumen_ejecutivo)
    st.subheader("Tres hallazgos clave")
    for h in mp.hallazgos:
        st.markdown(f"- {h}")
    izq, der = st.columns(2)
    izq.warning(mp.riesgos)
    der.info(mp.limitaciones)
    st.markdown(f"**Perfil de inversionista sugerido:** {mp.perfil_inversionista}")

    st.subheader("Ficha técnica")
    formatos = {k: "{:.2%}" for k in ficha_tecnica.index}
    formatos.update({"Sharpe": "{:.2f}", "Asimetría": "{:.2f}", "Exceso de curtosis": "{:.2f}"})
    st.dataframe(ficha_tecnica.apply(lambda fila: fila.map(formatos[fila.name].format), axis=1))

# =====================================================================
# 6. DATOS Y NOTA LEGAL
# =====================================================================
with st.expander("🔎 Ver y descargar los precios"):
    st.dataframe(cierre_periodo.tail(20))
    st.download_button("Descargar precios (CSV)", cierre_periodo.to_csv().encode("utf-8"),
                       "precios_portafolio.csv", "text/csv")

st.caption("⚠️ Rendimientos pasados no garantizan rendimientos futuros. Este análisis tiene "
           "fines académicos y no constituye una recomendación de inversión.")

Overwriting app.py


In [11]:
from pathlib import Path
Path("versiones/03_dashboard_final.py").write_text(Path("app.py").read_text(encoding="utf-8"), encoding="utf-8")
print("✔ Versión final guardada. Revisa el dashboard y prueba los controles.")

FileNotFoundError: [Errno 2] No such file or directory: 'versiones/03_dashboard_final.py'

### ✍️ Actividad de interpretación 💼

Usa **tu** dashboard para responder:

1. **Pestaña 💰:** completa la frase del cuadro azul. ¿Tu portafolio le ganó al benchmark?
2. **Pestaña ⚖️:** ¿el portafolio (★) queda a la izquierda de la mayoría de sus activos? ¿Qué significa?
3. **Pestaña 🔗:** ¿en qué porcentaje reduce el riesgo la diversificación? Explícalo en una frase sin tecnicismos.
4. **Pestaña 📉:** cambia el nivel de confianza de 95% a 99%. ¿Cuánto aumenta el VaR en dólares? ¿Qué VaR es mayor, el normal o el histórico? ¿Por qué?
5. **Período:** mueve el control de fechas para analizar solo el último año. ¿Cambia tu conclusión? ¿Qué te dice eso sobre los resultados históricos?

**Respuestas:**

## 07 · Personalizar el dashboard

### 🧩 7.1 Lo mínimo (obligatorio)

1. Completa en `mi_portafolio.py` **todos** los textos con tus resultados reales (sección 13 del taller). Cada afirmación debe tener una cifra que se vea en el dashboard.
2. Revisa que la pestaña **📝 Nuestro mensaje** se lea bien de principio a fin.

### 🧩 7.2 Colores del dashboard (opcional)

Streamlit lee los colores del archivo `.streamlit/config.toml`. Cambia los códigos de color y ejecuta la celda. **Este cambio necesita reiniciar el servidor:** vuelve a ejecutar la celda 4.1 (el enlace del túnel sigue siendo el mismo).

In [ ]:
from pathlib import Path
Path(".streamlit").mkdir(exist_ok=True)
Path(".streamlit/config.toml").write_text("""
[theme]
base = "light"
primaryColor = "#1f3b73"
backgroundColor = "#ffffff"
secondaryBackgroundColor = "#eef2f8"
textColor = "#1a1a1a"
""", encoding="utf-8")
print("✔ Tema guardado. Ejecuta de nuevo la celda 4.1 para aplicarlo.")

✔ Tema guardado. Ejecuta de nuevo la celda 4.1 para aplicarlo.


### 🧩 7.3 Retos (elige al menos uno)

Edita `app.py` desde el panel **Archivos**. Busca el bloque indicado y agrega el código.

**Reto A · Probabilidad de un mal día (taller, sección 8).** En la barra lateral, agrega:
```python
umbral = st.sidebar.slider("Umbral de pérdida diaria", -0.05, -0.01, -0.02, step=0.005, format="%.3f")
```
y en la pestaña 📉 muestra:
```python
prob = (rend_port <= umbral).mean()
st.metric(f"Días con pérdida mayor a {-umbral:.1%}", f"{prob:.2%}", f"≈ {prob * DIAS_ANIO:.0f} días al año", delta_color="off")
```

**Reto B · Gráfico Q-Q (taller, sección 9.5).** En la pestaña 📉, agrega la figura del taller cambiando `plt.show()` por `st.pyplot(fig)`.

**Reto C · Peso por activo.** En la pestaña 🧺, agrega `st.bar_chart(pesos)`.

**Reto D · Tu propio indicador.** Agrega una sexta métrica en la sección 4 (por ejemplo, la máxima caída `fp["Máxima caída"]`) usando `st.columns(6)`.

> Si algo se rompe, Streamlit muestra el error en rojo en la página, con la línea que falla. Léelo desde la última línea, igual que en Colab.

### Recuperar una versión anterior

Atención: reemplaza el contenido actual de `app.py`.

In [ ]:
from pathlib import Path
print("Versiones disponibles:")
for ruta in sorted(Path("versiones").glob("*.py")):
    print(" -", ruta)
# Para recuperar una versión, descomenta y adapta estas dos líneas:
# version = Path("versiones/03_dashboard_final.py")
# Path("app.py").write_text(version.read_text(encoding="utf-8"), encoding="utf-8")

Versiones disponibles:
 - versiones/01_esqueleto.py
 - versiones/02_datos_indicadores.py
 - versiones/03_dashboard_final.py


## 08 · Entregar y publicar

### 8.1 Archivos del proyecto

Para que otra persona (o Streamlit Community Cloud) pueda ejecutar tu dashboard se necesitan:

| Archivo | Para qué sirve |
|---|---|
| `app.py` | El dashboard |
| `mi_portafolio.py` | Tu ficha y tus textos |
| `precios_respaldo.csv` | Precios de respaldo si Yahoo Finance no responde |
| `requirements.txt` | Lista de librerías que se deben instalar |
| `README.md` | Descripción del proyecto |
| `.streamlit/config.toml` | Colores (solo si hiciste el paso 7.2) |

La carpeta `versiones/` es solo un respaldo de trabajo; no hace falta entregarla.

In [ ]:
from pathlib import Path
import mi_portafolio as mp, importlib
importlib.reload(mp)

Path("requirements.txt").write_text(
    "streamlit>=1.40,<2\npandas>=2.1\nnumpy>=1.26\nmatplotlib>=3.8\nseaborn>=0.13\nscipy>=1.11\nyfinance>=0.2.50\n",
    encoding="utf-8")

Path("README.md").write_text(f"""# {mp.nombre_portafolio}

Dashboard para inversionistas construido con Python y Streamlit
(curso Lenguajes de Programación · Ingeniería Financiera · ITM).

**Integrantes:** {mp.integrantes}

## Portafolio
Activos: {", ".join(mp.activos)} · Benchmark: {mp.benchmark} · Período: {mp.fecha_inicio} a {mp.fecha_fin}

## Ejecución local
```bash
pip install -r requirements.txt
streamlit run app.py
```

## Fuente de datos
Yahoo Finance mediante yfinance. Si la descarga falla, la app usa `precios_respaldo.csv` y lo indica en pantalla.

> Rendimientos pasados no garantizan rendimientos futuros. Uso académico; no es una recomendación de inversión.
""", encoding="utf-8")
print("✔ requirements.txt y README.md creados")

✔ requirements.txt y README.md creados


In [ ]:
import zipfile
from pathlib import Path

archivos = ["app.py", "mi_portafolio.py", "precios_respaldo.csv", "requirements.txt", "README.md"]
if Path(".streamlit/config.toml").exists():
    archivos.append(".streamlit/config.toml")

with zipfile.ZipFile("Dashboard_Portafolio.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for archivo in archivos:
        z.write(archivo)
print("✔ ZIP listo con:", archivos)

from google.colab import files
files.download("Dashboard_Portafolio.zip")

### 8.2 Publicar en Streamlit Community Cloud (enlace permanente)

El enlace de Cloudflare deja de funcionar cuando se cierra Colab. Para tener un enlace **permanente** que puedas compartir con el inversionista:

1. Crea un repositorio **público** en GitHub y sube los archivos del ZIP (descomprimidos). Para subir `.streamlit/config.toml`, usa **Add file → Create new file** y escribe `.streamlit/config.toml` como nombre.
2. Entra a [share.streamlit.io](https://share.streamlit.io/) con tu cuenta de GitHub.
3. **Create app →** elige el repositorio, la rama `main` y `app.py` como archivo principal.
4. Espera a que instale las librerías y copia el enlace de la app (termina en `.streamlit.app`).

> No confundas el enlace del **repositorio** (github.com/...) con el enlace de la **aplicación** (...streamlit.app). Si Yahoo Finance no responde desde la nube, la app usará `precios_respaldo.csv` y lo dirá en la barra lateral.

### 8.3 ✅ Lista de chequeo antes de entregar

- [ ] El dashboard abre sin errores y muestra **mi** portafolio (no el de ejemplo).
- [ ] Las cifras coinciden con las del taller para el mismo período.
- [ ] Todas las pestañas funcionan y los controles cambian los resultados.
- [ ] La fuente de los datos se ve en la barra lateral.
- [ ] La pestaña 📝 tiene resumen, tres hallazgos con cifras, riesgos, limitaciones y perfil.
- [ ] Se ve la nota legal al final de la página.
- [ ] Entregué el enlace del repositorio y el enlace de la app (o el ZIP y capturas).

### Criterios de evaluación sugeridos

| Criterio | Peso |
|---|---|
| El dashboard funciona y muestra el portafolio propio | 25% |
| Indicadores y gráficos correctos y coherentes con el taller | 20% |
| Mensaje para el inversionista: claro, con cifras y limitaciones | 25% |
| Personalización (colores, al menos un reto) | 15% |
| Publicación y archivos para reproducir el proyecto | 15% |

---
> ⚠️ *Rendimientos pasados no garantizan rendimientos futuros. Este análisis tiene fines académicos y no constituye una recomendación de inversión.*